# Hands-on Lab — Getting Practical with LLMs: Fine-Tuning BERT

*Statistical Foundations of LLMs · Session 2 · Accompanies slides 192–199*

Time to go from theory to practice. You will take a **pretrained BERT** model, fine-tune it on movie-review sentiment (`rotten_tomatoes`), and test it on your own sentences — experiencing first-hand why *transfer learning* changed NLP.

> ⚡ **This is the one notebook that wants a GPU.** Section 2 below shows you exactly how to switch it on in Colab (about 15 seconds) and how to confirm it worked. On a GPU the training cell takes ~2–3 minutes; on CPU it still runs if you use the smaller subsets noted later.

> **EXECUTED SOLUTIONS NOTEBOOK.** Pre-run instructor copy with all code executed and outputs saved. Source: `notebooks/solutions/`. Regenerate with `python scripts/export_executed_solutions.py`.


> ✅ **SOLUTIONS NOTEBOOK.** Every exercise stub is filled in and every challenge cell contains working reference code. Use this for self-check or as an instructor key — encourage students to attempt the exercises in the main notebook first.

## Learning Objectives

1. Load and inspect a dataset with 🤗 `datasets`.
2. Tokenize text for BERT: `input_ids`, `attention_mask`, truncation, padding.
3. Load `BertForSequenceClassification` and understand what the "classification head" is.
4. Fine-tune with the 🤗 `Trainer` API and track evaluation metrics.
5. Run inference on your own sentences and diagnose generalization vs. overfitting.


## 0. New to Jupyter? Start Here (2 minutes)

**What is a Jupyter Notebook?** A document that mixes text and runnable Python code, organized in *cells*.

| What you need to know | How |
|---|---|
| **Run a cell** | Click it, then press **Shift + Enter** (or the ▶ button) |
| **Cell types** | **Markdown** cells = formatted text (like this one). **Code** cells = Python you can execute |
| **Order matters** | Run cells **top to bottom**. A cell may depend on variables defined above it |
| **Restart the kernel** | Menu: *Runtime → Restart runtime* (Colab) or *Kernel → Restart* (Jupyter). Then re-run cells from the top |
| **Install packages** | Run a cell starting with `%pip install ...`, then restart the kernel if asked |
| **Modify code** | Just edit any code cell and re-run it — experimenting is the whole point! |
| **Read outputs** | Results appear directly below each code cell: printed text, tables, or plots |

> 💡 **Tip:** If something behaves strangely, *Restart runtime* and run all cells from the top (*Runtime → Run all*).


## 1. Background: Why Fine-Tune?

BERT was pretrained on ~3.3B words with **Masked Language Modeling** (predict hidden words) and Next Sentence Prediction. That pretraining taught it grammar, word meaning, and world knowledge — but *nothing* about our sentiment task.

**Fine-tuning** adds a small classification layer on top and updates all weights on our labeled data. Because the language knowledge is already there, a few thousand examples and a couple of epochs suffice — versus millions of examples to train from scratch.

$$\underbrace{\text{pretraining}}_{\text{expensive, once, unlabeled data}} \;\to\; \underbrace{\text{fine-tuning}}_{\text{cheap, per-task, small labeled data}}$$


## 2. Setup and Imports

### 2.0 Turn on the free GPU first (one time, ~15 seconds)

A **GPU** makes the training in this notebook about 10× faster. In Colab:

1. In the top menu, click **Runtime → Change runtime type**.
2. Under **Hardware accelerator**, choose **T4 GPU**.
3. Click **Save**. Colab may reconnect — that is normal.

Then run the check cell below. It should say **"GPU is ON"**. If it says CPU, just repeat the three steps above (and make sure to click Save).

> No GPU available today? No problem — the notebook still works on CPU. Later, where you see `N_TRAIN, N_EVAL = 2000, 500`, change it to `200, 50` so training stays fast.


In [1]:
import torch

if torch.cuda.is_available():
    print("✅ GPU is ON —", torch.cuda.get_device_name(0))
else:
    print("⚠️ Running on CPU (no GPU). This still works; use the smaller "
          "N_TRAIN/N_EVAL noted later so it stays fast.")


✅ GPU is ON — NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
# Colab usually needs this once (then restart if prompted):
# %pip install -U transformers datasets evaluate accelerate --quiet


In [3]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cuda


## 3. Step 1 — Load and Explore the Dataset

`rotten_tomatoes`: 10,662 movie-review sentences labeled positive (1) or negative (0). **Always look at your data before modeling it.**


In [4]:
dataset = load_dataset("rotten_tomatoes")
print(dataset)


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a6590fa6-a7ca-4526-ae69-169825a66cbf)')' thrown while requesting HEAD https://huggingface.co/datasets/rotten_tomatoes/resolve/main/README.md


Retrying in 1s [Retry 1/5].


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


In [5]:
import random

for i in random.Random(0).sample(range(len(dataset["train"])), 3):
    ex = dataset["train"][i]
    label = "POSITIVE" if ex["label"] == 1 else "NEGATIVE"
    print(f"[{label}] {ex['text']}\n")

labels = np.array(dataset["train"]["label"])
print(f"Class balance: {labels.mean():.1%} positive — perfectly balanced by design.")


[NEGATIVE] unlike his directorial efforts , la femme nikita and the professional , the transporter lacks besson's perspective as a storyteller .

[NEGATIVE] 'dragonfly' is a movie about a bus wreck that turns into a film wreck .

[POSITIVE] look , this is a terrific flick replete with dazzling camera-work , dancing and music .

Class balance: 50.0% positive — perfectly balanced by design.


## 4. Step 2 — Tokenize and Preprocess

BERT can't read text; it reads **token IDs**. The tokenizer also returns an **attention mask** (1 = real token, 0 = padding) so the model ignores the padding we add to make every sequence length 128.


In [6]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def preprocess(example):
    return tokenizer(
        example["text"],
        truncation=True,          # cut sequences longer than max_length
        padding="max_length",     # pad shorter ones
        max_length=128,
    )

tokenized = dataset.map(preprocess, batched=True)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
print(tokenized["train"])


D:\dev\llm\llm-statistical-foundations-course\.venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8530
})


### 👀 Peek inside one tokenized example


In [7]:
sample = tokenized["train"][0]
ids = sample["input_ids"]
n_real = int(sample["attention_mask"].sum())

print("Original: ", dataset["train"][0]["text"][:80], "...")
print("Tokens:   ", tokenizer.convert_ids_to_tokens(ids[:12]), "...")
print("input_ids:", ids[:12].tolist(), "...")
print(f"attention_mask: {n_real} real tokens, {128 - n_real} padding")


Original:  the rock is destined to be the 21st century's new " conan " and that he's going  ...
Tokens:    ['[CLS]', 'the', 'rock', 'is', 'destined', 'to', 'be', 'the', '21st', 'century', "'", 's'] ...
input_ids: [101, 1996, 2600, 2003, 16036, 2000, 2022, 1996, 7398, 2301, 1005, 1055] ...
attention_mask: 47 real tokens, 81 padding


### ✏️ Exercise 4.1

Tokenize the word `"unbelievably"` and the (fake) word `"biostatisticsful"` with `tokenizer.tokenize(...)`. What do the `##` pieces mean? Why does subword tokenization mean BERT never sees an "unknown word"?


In [8]:
print(tokenizer.tokenize("unbelievably"))
print(tokenizer.tokenize("biostatisticsful"))
# The '##' pieces are subword continuations: WordPiece breaks rare/unknown words
# into known sub-pieces (e.g. 'bio', '##sta', '##tist', ...). Because any word can
# be spelled from subwords, BERT never emits a true [UNK] — this is why subword
# tokenization gives open-vocabulary coverage.

['un', '##bel', '##ie', '##va', '##bly']
['bio', '##sta', '##tist', '##ics', '##ful']


## 5. Step 3 — Load the Pretrained Model

Note the warning that appears: the classification head is **newly initialized** — those weights are random and are exactly what fine-tuning will learn.


In [9]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params/1e6:.0f}M  (all will be updated during fine-tuning)")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parameters: 109M  (all will be updated during fine-tuning)


## 6. Step 4 — Fine-Tune with the Trainer API

We train on a subset to keep class time short. **GPU:** 2,000 train / 500 eval ≈ 2–3 min. **CPU only:** change to 200 / 50.


In [10]:
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


Using the latest cached version of the module from C:\Users\Maods\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--accuracy\f887c0aab52c2d38e1f8a215681126379eca617f96c447638f751434e8e65b14 (last modified on Thu Jul  9 12:09:29 2026) since it couldn't be found locally at evaluate-metric--accuracy, or remotely on the Hugging Face Hub.


In [11]:
N_TRAIN, N_EVAL = 2000, 500   # CPU-only? use 200, 50

args = TrainingArguments(
    output_dir="./bert-rotten",
    eval_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=2,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"].shuffle(seed=42).select(range(N_TRAIN)),
    eval_dataset=tokenized["validation"].select(range(N_EVAL)),
    compute_metrics=compute_metrics,
)

trainer.train()


  0%|          | 0/250 [00:00<?, ?it/s]

{'loss': 0.467, 'grad_norm': 11.859583854675293, 'learning_rate': 2.5e-05, 'epoch': 1.0}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.4307303726673126, 'eval_accuracy': 0.804, 'eval_runtime': 2.0731, 'eval_samples_per_second': 241.183, 'eval_steps_per_second': 30.389, 'epoch': 1.0}


{'loss': 0.1639, 'grad_norm': 14.880958557128906, 'learning_rate': 0.0, 'epoch': 2.0}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.39401549100875854, 'eval_accuracy': 0.872, 'eval_runtime': 2.0898, 'eval_samples_per_second': 239.255, 'eval_steps_per_second': 30.146, 'epoch': 2.0}
{'train_runtime': 55.471, 'train_samples_per_second': 72.11, 'train_steps_per_second': 4.507, 'train_loss': 0.3154573745727539, 'epoch': 2.0}


TrainOutput(global_step=250, training_loss=0.3154573745727539, metrics={'train_runtime': 55.471, 'train_samples_per_second': 72.11, 'train_steps_per_second': 4.507, 'total_flos': 263111055360000.0, 'train_loss': 0.3154573745727539, 'epoch': 2.0})

In [12]:
results = trainer.evaluate()
print(f"Validation accuracy: {results['eval_accuracy']:.1%}")


  0%|          | 0/63 [00:00<?, ?it/s]

Validation accuracy: 87.2%


**Reading the numbers.** With 2,000 examples and 2 epochs you should see roughly **83–88%** validation accuracy (~92% is reachable with the full training set). Compare train loss vs. eval loss across epochs:

- eval loss ↓ together with train loss → **generalizing**
- train loss ↓ but eval loss ↑ → **overfitting**
- both high and flat → **underfitting** (train longer / more data)


## 7. Step 5 — Run Inference on Your Own Sentences


In [13]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding="max_length", max_length=128).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred = int(torch.argmax(probs))
    label = "POSITIVE" if pred == 1 else "NEGATIVE"
    return f"{label}  (confidence {probs[pred]:.1%})"

print(predict_sentiment("I loved the cinematography and acting!"))
print(predict_sentiment("A tedious, predictable waste of two hours."))


POSITIVE  (confidence 99.2%)
NEGATIVE  (confidence 99.3%)


### ✏️ Exercise 7.1 — Break the model

Find sentences the model gets **wrong** or is **unsure** about. Try:

1. Negation: *"not bad at all"*
2. Sarcasm: *"oh great, another superhero movie"*
3. Mixed sentiment: *"brilliant acting, terrible script"*
4. Out-of-domain text: a sentence about a **restaurant** or a **research paper**

Record the confidence for each. Where does it fail, and can you explain why from the training data?


In [14]:
tests = [
    "not bad at all",                         # negation
    "oh great, another superhero movie",      # sarcasm
    "brilliant acting, terrible script",      # mixed
    "the enzyme assay produced stable results",  # out-of-domain
]
for t in tests:
    print(f"{t:45} -> {predict_sentiment(t)}")
# Typical findings: negation and sarcasm lower confidence or flip the label,
# because the model leans on lexical cues ('great', 'terrible') that context can
# invert. Out-of-domain text yields low-confidence, unreliable predictions.

not bad at all                                -> NEGATIVE  (confidence 95.5%)
oh great, another superhero movie             -> POSITIVE  (confidence 85.0%)
brilliant acting, terrible script             -> NEGATIVE  (confidence 98.4%)
the enzyme assay produced stable results      -> POSITIVE  (confidence 96.5%)


## 8. Reflection Questions

Answer briefly (these mirror slides 194 & 199); model answers are collapsed below.

1. **Dataset & preprocessing:** What are `input_ids` and `attention_mask`, and why are both needed?
2. **Pretrained model:** What tasks was BERT originally pretrained on? Why does that help sentiment classification?
3. **Fine-tuning:** Did your model overfit, underfit, or generalize? What evidence do you have?
4. **Inference:** Report one surprising prediction from Exercise 7.1 and explain it.
5. **Strategy:** For adapting to a *new domain* (e.g., clinical trial reports), would you choose full fine-tuning, prompt engineering, or RAG? Why?


<details><summary>💡 <b>Model answers (expand after writing your own)</b></summary>

1. `input_ids` = integer indices of subword tokens; `attention_mask` = 1/0 flags so self-attention ignores padding positions. Without the mask, padding would contaminate the representation.
2. Masked Language Modeling + Next Sentence Prediction. MLM forces the model to learn syntax, semantics, and world knowledge — general language competence our task reuses for free.
3. Expected: mild generalization — train and eval loss both decrease; eval accuracy ~85%. Overfitting appears if you train many epochs on the small subset (train loss → 0, eval loss rises).
4. Typical finding: negation and sarcasm flip labels or drop confidence — the model relies partly on lexical cues ("great", "waste") that context can invert.
5. Reasonable answers: domain-specific fine-tuning (e.g., start from BioBERT/ClinicalBERT) with labeled data; prompting or RAG when labels are scarce or facts must be current. There is no single right answer — justify the trade-off (data, cost, freshness).
</details>


## 9. 🏆 Challenge Exercises

**Challenge A — DistilBERT race.** Swap in `distilbert-base-uncased` (use `AutoTokenizer` / `AutoModelForSequenceClassification`). Compare accuracy and training time. Is 40% faster worth ~1–2 accuracy points?

**Challenge B — Learning curve.** Fine-tune with 100 / 500 / 2,000 / 8,530 training examples and plot accuracy vs. dataset size. Where are the diminishing returns? *(This is the statistics of transfer learning in one plot.)*

**Challenge C — Frozen features.** Freeze all BERT layers except the classifier head (`for p in model.bert.parameters(): p.requires_grad = False`). How much accuracy do you lose? What does that say about where task knowledge lives?


In [15]:
# --- Challenge A: DistilBERT speed/accuracy comparison ---
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import time

d_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
d_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

d_args = TrainingArguments(output_dir="./distil-rotten", eval_strategy="epoch",
                           per_device_train_batch_size=16, num_train_epochs=2,
                           report_to="none", seed=42)

def tok_distil(ex):
    return d_tok(ex["text"], truncation=True, padding="max_length", max_length=128)

d_tokenized = dataset.map(tok_distil, batched=True)
d_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

d_trainer = Trainer(model=d_model, args=d_args,
                    train_dataset=d_tokenized["train"].shuffle(seed=42).select(range(N_TRAIN)),
                    eval_dataset=d_tokenized["validation"].select(range(N_EVAL)),
                    compute_metrics=compute_metrics)
t0 = time.time(); d_trainer.train(); print(f"DistilBERT trained in {time.time()-t0:.0f}s")
print("DistilBERT accuracy:", d_trainer.evaluate()["eval_accuracy"])
# DistilBERT is ~40% faster and smaller, usually within 1-2 accuracy points of BERT.

# --- Challenge B: learning curve (sketch) ---
# Loop N in [100, 500, 2000, 8530], retrain, record eval accuracy, then:
#   plt.plot(sizes, accuracies); accuracy rises fast then plateaus — diminishing
#   returns are the statistics of transfer learning in one picture.

# --- Challenge C: frozen features ---
# for p in model.bert.parameters(): p.requires_grad = False
# Freezing the encoder and training only the classifier head keeps most of the
# accuracy, showing that BERT's pretrained representations already encode the
# features; fine-tuning mostly adapts the small head.

D:\dev\llm\llm-statistical-foundations-course\.venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.6061309576034546, 'eval_accuracy': 0.714, 'eval_runtime': 1.1709, 'eval_samples_per_second': 427.029, 'eval_steps_per_second': 53.806, 'epoch': 1.0}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.4952891170978546, 'eval_accuracy': 0.81, 'eval_runtime': 1.1924, 'eval_samples_per_second': 419.337, 'eval_steps_per_second': 52.837, 'epoch': 2.0}
{'train_runtime': 30.4417, 'train_samples_per_second': 131.399, 'train_steps_per_second': 8.212, 'train_loss': 0.36984780883789065, 'epoch': 2.0}
DistilBERT trained in 31s


  0%|          | 0/63 [00:00<?, ?it/s]

DistilBERT accuracy: 0.81


## Key Takeaways

- **Pretrain once, fine-tune everywhere**: a few thousand labeled examples + 2 epochs approx strong performance, because language knowledge transfers.
- The tokenizer's outputs (`input_ids`, `attention_mask`) are the contract between raw text and the transformer.
- Always inspect **train vs. eval curves** — accuracy alone hides overfitting.
- Models inherit their training data's blind spots: negation, sarcasm, and domain shift are where fine-tuned models break.

## References

- Devlin et al. (2019), *BERT: Pre-training of Deep Bidirectional Transformers* — https://arxiv.org/abs/1810.04805
- HF fine-tuning tutorial — https://huggingface.co/docs/transformers/training
